# V1-14: LLM 기업명 검색어 제안 프롬프트 개선 평가

13번에서는 Qwen이 불확실한 입력에 거의 항상 빈 목록을 반환해 후보 회수 이득이 없었다. 이번에는 **같은 로컬 Qwen 모델**에서 프롬프트의 역할과 보류 정책만 바꿔, 명백한 약칭·오타에는 검색 가설을 제안하고 포괄·대상 부재 표현은 보류할 수 있는지 평가한다.

> LLM은 `corp_code`를 선택하지 않으며, 검색어 가설만 반환한다. OpenDART 검증과 최종 재질문·확정 정책은 코드의 책임이다.

## 1. 비교 목적

- **A. deterministic 기준선**: 원문 표현으로 OpenDART exact + fuzzy 후보를 만든다.
- **B. 13번 보수 프롬프트**: 이전 실행 결과를 역사적 기준으로만 참고한다. 같은 모델의 재호출은 하지 않는다.
- **C. 개선 프롬프트**: 고유 raw exact가 없는 경우, 명백한 약칭·오타·표기 차이에는 최대 2개의 정식 기업명 검색 가설을 제안한다. 포괄 표현·동명 충돌·대상 부재는 보류한다.

13번 B는 Candidate Recall 0.762, LLM 신규 기여 0, 호출 p50 52.09초였다. 이번 실행 결과 C를 A 및 그 역사적 B와 탐색적으로 비교한다. C는 13번과 동일한 평가 입력·모델·temperature를 사용하지만, 실행 시점의 OpenDART 목록·모델 런타임이 다를 수 있으므로 B/C 차이를 프롬프트 효과로 단정하지 않는다. 최종 채택 전에는 같은 데이터 스냅샷·실행 환경에서 B/C를 재실행한다.

## 2. 평가 지표와 우선순위

품질·안전성 지표를 우선으로 본다. 로컬 모델의 지연시간은 기록하지만, 이 단계에서 모델·프롬프트의 품질이 명확히 좋다면 지연시간만으로 미채택하지 않는다. 운영 모델(API 포함) 비교 단계에서 비용·속도를 다시 비교한다.

- **Candidate Recall**: 정답 기업이 최종 후보 집합에 포함됐는가
- **LLM Candidate Recall**: LLM이 직접 제안한 검색어 후보 안에 정답 기업이 포함됐는가
- **LLM Expansion Contribution**: 기준선이 놓친 정답을 LLM이 새로 후보에 넣었는가
- **Abstention Accuracy**: 포괄 표현·대상 부재에서 LLM이 후보를 만들지 않고 보류했는가
- **Structured-output Success Rate / Search-term Validity**: 구조와 검색어가 실제 후속 코드에 쓸 수 있는가
- **LLM Call Rate / p50·p95**: 참고용 운영 측정값


In [1]:
import io
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from math import ceil
import os
from pathlib import Path
import re
from statistics import median
import sys
import time
from typing import Literal
import xml.etree.ElementTree as ET
import zipfile

from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from rapidfuzz import fuzz, process

/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
MODEL_NAME = os.environ.get('OLLAMA_MODEL', 'qwen3.6:27b')
TEMPERATURE = 0
PROMPT_VERSION = 'entity-search-plan-v2'

print(f'프로젝트 루트: {project_root}')
print(f'모델: {MODEL_NAME} / temperature={TEMPERATURE}')

프로젝트 루트: /home/sms/openclaw_file/FinSight
모델: qwen3.6:27b / temperature=0


In [3]:
sys.path.insert(0, str(project_root / 'src'))

from dart.client import CORP_CODE_URL, _get_bytes


def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    zip_stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(zip_stream):
        raise RuntimeError(zip_bytes.decode('utf-8', errors='replace'))
    with zipfile.ZipFile(zip_stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])
    root = ET.fromstring(xml_bytes)
    return [{child.tag: child.text or '' for child in item} for item in root.findall('list')]


corp_records = load_corp_records(api_key)
retrieved_at = datetime.now().astimezone().isoformat(timespec='seconds')
print(f'OpenDART 기업코드 조회 시각: {retrieved_at}')
print(f'전체 법인 레코드 수: {len(corp_records):,}')

OpenDART 기업코드 조회 시각: 2026-09-14T17:27:52+09:00
전체 법인 레코드 수: 119,183


In [4]:
def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())


def build_name_index(records):
    index = {}
    for record in records:
        for name in {record['corp_name'], record['corp_eng_name']}:
            key = normalize_name(name)
            if key:
                index.setdefault(key, {})[record['corp_code']] = record
    return {key: list(values.values()) for key, values in index.items()}


name_index = build_name_index(corp_records)
name_keys = list(name_index)

TARGETS = {
    '삼성전자': ('삼성전자', '005930'), 'SK하이닉스': ('SK하이닉스', '000660'),
    'LG전자': ('LG전자', '066570'), '현대자동차': ('현대자동차', '005380'),
    'NAVER': ('NAVER', '035420'), '카카오': ('카카오', '035720'),
    'POSCO홀딩스': ('POSCO홀딩스', '005490'), '삼성바이오로직스': ('삼성바이오로직스', '207940'),
    '셀트리온': ('셀트리온', '068270'), '기아': ('기아', '000270'),
}

target_records = {}
for key, (official_name, stock_code) in TARGETS.items():
    matches = [record for record in corp_records if normalize_name(record['corp_name']) == normalize_name(official_name) and record['stock_code'].strip() == stock_code]
    if len(matches) != 1:
        raise AssertionError(f'{key}: OpenDART 기준 법인 검증 실패 ({len(matches)}건)')
    target_records[key] = matches[0]

print(f'정규화 이름 키 수: {len(name_keys):,}')

정규화 이름 키 수: 219,425


## 3. 고정 평가 사례

`기대 대상`과 `LLM 기대 행동`은 채점에만 쓰며 LLM 프롬프트로 전달하지 않는다. `candidate`는 검색 가설을 제안해도 되는 사례, `abstain`은 포괄 표현 또는 대상 부재이므로 보류해야 하는 사례다. 이는 최종 자동 확정 정답이 아니다.

In [5]:
CASES = [
    ('삼성전자 2024년 매출을 알려줘.', '삼성전자', ['삼성전자'], 'skip'),
    ('삼성 전자 실적을 요약해줘.', '삼성 전자', ['삼성전자'], 'skip'),
    ('삼전의 최근 매출 추세를 보여줘.', '삼전', ['삼성전자'], 'candidate'),
    ('삼성전쟈 영업이익을 알려줘.', '삼성전쟈', ['삼성전자'], 'candidate'),
    ('SK하이닉스의 사업보고서 기준 매출은?', 'SK하이닉스', ['SK하이닉스'], 'skip'),
    ('sk 하이닉스 실적을 알려줘.', 'sk 하이닉스', ['SK하이닉스'], 'skip'),
    ('하이닉스 매출 추세를 보고 싶어.', '하이닉스', ['SK하이닉스'], 'candidate'),
    ('하닉 실적을 정리해줘.', '하닉', ['SK하이닉스'], 'candidate'),
    ('LG전자 매출을 알려줘.', 'LG전자', ['LG전자'], 'skip'),
    ('엘지전자 영업이익 추이를 알려줘.', '엘지전자', ['LG전자'], 'candidate'),
    ('현대자동차의 최근 실적은?', '현대자동차', ['현대자동차'], 'skip'),
    ('현대차 매출을 알려줘.', '현대차', ['현대자동차'], 'candidate'),
    ('현차 최근 매출 추세는?', '현차', ['현대자동차'], 'candidate'),
    ('NAVER의 매출을 알려줘.', 'NAVER', ['NAVER'], 'skip'),
    ('네이버 영업이익을 알려줘.', '네이버', ['NAVER'], 'candidate'),
    ('카카오의 매출을 알려줘.', '카카오', ['카카오'], 'candidate'),
    ('POSCO홀딩스 실적을 분석해줘.', 'POSCO홀딩스', ['POSCO홀딩스'], 'skip'),
    ('포스코홀딩스 영업이익을 보여줘.', '포스코홀딩스', ['POSCO홀딩스'], 'candidate'),
    ('삼바의 최근 실적은?', '삼바', ['삼성바이오로직스'], 'candidate'),
    ('셀트 매출 추세를 알려줘.', '셀트', ['셀트리온'], 'candidate'),
    ('기아차의 영업이익을 알려줘.', '기아차', ['기아'], 'candidate'),
    ('엘지 계열사의 실적을 알려줘.', '엘지', [], 'abstain'),
    ('현대 실적을 알려줘.', '현대', [], 'abstain'),
    ('감자우주물산2026의 매출을 알려줘.', '감자우주물산2026', [], 'abstain'),
    ('BLUEORBITVENTURES2049의 실적을 알려줘.', 'BLUEORBITVENTURES2049', [], 'abstain'),
    ('QXZ테스트법인9999 재무정보를 알려줘.', 'QXZ테스트법인9999', [], 'abstain'),
]

TEST_CASES = [
    {'질문': question, '원문 기업 표현': raw_name, '기대 대상': targets, 'LLM 기대 행동': expectation}
    for question, raw_name, targets, expectation in CASES
]

print(f'전체 사례: {len(TEST_CASES)}개')
print(f'LLM 후보 제안 기대 사례: {sum(row["LLM 기대 행동"] == "candidate" for row in TEST_CASES)}개')
print(f'LLM 보류 기대 사례: {sum(row["LLM 기대 행동"] == "abstain" for row in TEST_CASES)}개')

전체 사례: 26개
LLM 후보 제안 기대 사례: 13개
LLM 보류 기대 사례: 5개


In [6]:
FUZZY_KEY_LIMIT = 50
FUZZY_RESULT_LIMIT = 5


def exact_records(query):
    return name_index.get(normalize_name(query), [])


def fuzzy_records(query):
    scored_keys = process.extract(normalize_name(query), name_keys, scorer=fuzz.ratio, limit=FUZZY_KEY_LIMIT)
    best_by_code = {}
    for key, score, _ in scored_keys:
        for record in name_index[key]:
            if record['corp_code'] not in best_by_code or score > best_by_code[record['corp_code']][1]:
                best_by_code[record['corp_code']] = (record, score)
    return sorted(best_by_code.values(), key=lambda item: (item[1], bool(item[0]['stock_code'].strip()), item[0]['modify_date']), reverse=True)[:FUZZY_RESULT_LIMIT]


def union_candidates(query, source_prefix):
    candidates = {}
    for record in exact_records(query):
        candidates[record['corp_code']] = {'record': record, 'sources': {f'{source_prefix}_exact'}}
    for record, score in fuzzy_records(query):
        current = candidates.setdefault(record['corp_code'], {'record': record, 'sources': set()})
        current['sources'].add(f'{source_prefix}_fuzzy')
    return candidates


def merge_candidates(*candidate_sets):
    merged = {}
    for candidate_set in candidate_sets:
        for corp_code, candidate in candidate_set.items():
            current = merged.setdefault(corp_code, {'record': candidate['record'], 'sources': set()})
            current['sources'].update(candidate['sources'])
    return merged


def needs_llm(raw_name):
    return len(exact_records(raw_name)) != 1

print('실제 LLM 호출 대상:', sum(needs_llm(case['원문 기업 표현']) for case in TEST_CASES))

실제 LLM 호출 대상: 18


## 4. 개선된 Entity Search Plan

이 프롬프트는 특정 기업명 예시를 주지 않는다. 대신 원문이 약칭·오타·표기 차이로 보이면 검색 가설을 제안하고, 가설은 코드가 검증하며 자동 확정되지 않는다는 역할 경계를 명시한다.

In [7]:
class EntitySearchPlan(BaseModel):
    decision: Literal['candidate', 'ambiguous', 'unknown'] = Field(
        description='검색 가설 제안, 포괄·다의적 표현, 또는 해석 불가 상태'
    )
    search_terms: list[str] = Field(
        default_factory=list, max_length=2,
        description='OpenDART에서 검증할 정식 기업명 검색 가설. corp_code·종목코드·설명 문장은 금지.'
    )


SYSTEM_PROMPT = '''
당신은 한국 기업 리서치 시스템의 기업명 검색 계획기다.
프로그램은 원문 기업 표현을 OpenDART에서 고유하게 exact 확정하지 못했을 때만 당신을 호출한다.

원문이 명백한 약칭, 한글·영문 표기 차이, 또는 가벼운 철자 오타로 보이면 decision='candidate'와 함께
정식 기업명으로 보이는 검색 가설을 최대 2개 제안한다. 완전한 확신이 없어도 검색 가설은 제안할 수 있다.
이 가설은 정답 확정이 아니며, 뒤의 프로그램이 OpenDART 기업 목록으로 검증하고 필요하면 사용자에게 재질문한다.

원문이 포괄적인 그룹명, 여러 기업을 뜻할 수 있는 표현, 또는 실제 기업으로 해석할 근거가 없는 표현이면
decision='ambiguous' 또는 decision='unknown'을 반환하고 search_terms=[]로 둔다.

사용자가 언급하지 않은 계열사·경쟁사·관련 기업을 추가하지 않는다.
corp_code, 종목코드, 투자 의견, 설명 문장을 반환하지 않는다.
'''

llm = ChatOllama(model=MODEL_NAME, base_url=OLLAMA_BASE_URL, temperature=TEMPERATURE)
planner = llm.with_structured_output(EntitySearchPlan, include_raw=True)

RUN_CONFIG = {
    'run_at_utc': datetime.now(timezone.utc).isoformat(),
    'model': MODEL_NAME, 'temperature': TEMPERATURE, 'prompt_version': PROMPT_VERSION,
    'langchain_version': version('langchain'),
    'langchain_ollama_version': version('langchain-ollama'),
}
RUN_CONFIG

{'run_at_utc': '2026-09-14T08:28:17.274443+00:00',
 'model': 'qwen3.6:27b',
 'temperature': 0,
 'prompt_version': 'entity-search-plan-v2',
 'langchain_version': '1.2.10',
 'langchain_ollama_version': '1.1.0'}

In [ ]:
def build_messages(question, raw_name):
    return [
        ('system', SYSTEM_PROMPT),
        ('human', f'사용자 질문: {question}\n원문 기업 표현: {raw_name}'),
    ]


def raw_terms(plan):
    return list(plan.search_terms) if plan is not None else []


def clean_terms(plan):
    if plan is None:
        return [], []
    if plan.decision != 'candidate':
        return [], raw_terms(plan)
    terms, rejected, seen = [], [], set()
    for term in plan.search_terms:
        term = term.strip()
        key = normalize_name(term)
        if not term or len(term) > 80 or not key or key.isdigit() or key in seen:
            rejected.append(term)
            continue
        seen.add(key)
        terms.append(term)
    return terms[:2], rejected


print('[사전 호출 1/1] 원문 기업 표현: 하닉 / Qwen 호출 시작', flush=True)
smoke_started_at = time.perf_counter()
try:
    smoke = planner.invoke(build_messages('하닉 실적을 정리해줘.', '하닉'))
    smoke_elapsed_ms = (time.perf_counter() - smoke_started_at) * 1_000
    print(f'[사전 호출 1/1] 완료 / {smoke_elapsed_ms:,.0f}ms', flush=True)
    print('파싱 결과:', smoke['parsed'])
    print('파싱 오류:', smoke['parsing_error'])
    print('정리된 검색어:', clean_terms(smoke['parsed'])[0])
except Exception as error:
    smoke_elapsed_ms = (time.perf_counter() - smoke_started_at) * 1_000
    print(f'[사전 호출 1/1] 실패 / {smoke_elapsed_ms:,.0f}ms', flush=True)
    print(f'단일 호출 확인 실패: {type(error).__name__}: {error}')

## 5. C 후보 실행

이 셀은 고유 raw exact가 아닌 입력에만 모델을 한 번씩 호출한다. 호출 오류는 한 사례의 실패로 기록하고 다른 사례 집계는 계속한다. LLM이 낸 검색어는 exact·fuzzy 후보로 다시 탐색하지만, LLM 제안만으로 최종 기업을 확정하지 않는다.

In [ ]:
rows = []
llm_total = sum(needs_llm(case['원문 기업 표현']) for case in TEST_CASES)
llm_number = 0
print(f'전체 평가 시작: LLM 호출 {llm_total}회, raw exact 고유 일치 입력은 건너뜁니다.', flush=True)

for case_number, case in enumerate(TEST_CASES, start=1):
    raw_name = case['원문 기업 표현']
    baseline = union_candidates(raw_name, 'raw')
    llm_called = needs_llm(raw_name)
    plan = None
    error = None
    raw_search_terms, proposed_terms, rejected_terms, llm_latency_ms = [], [], [], None
    llm_candidates = {}

    if llm_called:
        llm_number += 1
        print(f'[LLM {llm_number}/{llm_total}] 사례 {case_number}/{len(TEST_CASES)} / {raw_name!r} / 호출 시작', flush=True)
        started_at = time.perf_counter()
        try:
            result = planner.invoke(build_messages(case['질문'], raw_name))
            plan = result['parsed']
            error = result['parsing_error']
            raw_search_terms = raw_terms(plan)
            proposed_terms, rejected_terms = clean_terms(plan)
            llm_candidates = merge_candidates(*(union_candidates(term, 'llm') for term in proposed_terms))
        except Exception as exc:
            error = f'{type(exc).__name__}: {exc}'
        finally:
            llm_latency_ms = (time.perf_counter() - started_at) * 1_000
            status = '완료' if error is None and plan is not None else '실패'
            print(f'[LLM {llm_number}/{llm_total}] {status} / {llm_latency_ms:,.0f}ms', flush=True)
    else:
        print(f'[건너뜀] 사례 {case_number}/{len(TEST_CASES)} / {raw_name!r} / 고유 raw exact', flush=True)

    expanded = merge_candidates(baseline, llm_candidates)
    expected_codes = {target_records[key]['corp_code'] for key in case['기대 대상']}
    baseline_codes, llm_codes, expanded_codes = set(baseline), set(llm_candidates), set(expanded)
    term_validity = [bool(term.strip()) and bool(exact_records(term.strip())) for term in raw_search_terms]
    is_safe_abstention = (
        plan is not None and plan.decision in {'ambiguous', 'unknown'} and not raw_search_terms
    )
    rows.append({
        **case,
        'LLM 호출': llm_called,
        'LLM 결정': plan.decision if plan else None,
        'LLM 원본 검색어': raw_search_terms,
        '실제 탐색 검색어': proposed_terms,
        '거부된 검색어': rejected_terms,
        '구조화 출력 성공': None if not llm_called else error is None and plan is not None,
        '원본 검색어 exact 유효 여부': term_validity,
        '기준선 정답 포함': bool(expected_codes & baseline_codes) if expected_codes else None,
        'LLM 정답 포함': bool(expected_codes & llm_codes) if expected_codes else None,
        '확장 후 정답 포함': bool(expected_codes & expanded_codes) if expected_codes else None,
        'LLM 신규 기여': bool((expected_codes & llm_codes) - baseline_codes) if expected_codes else None,
        '안전 보류': is_safe_abstention if case['LLM 기대 행동'] == 'abstain' else None,
        '기준선 후보 수': len(baseline), 'LLM 후보 수': len(llm_candidates), '확장 후 후보 수': len(expanded),
        'LLM 호출 시간(ms)': llm_latency_ms, '오류': str(error) if error else None,
    })

print(f'전체 평가 완료: 사례 {len(rows)}개, LLM 호출 {llm_number}회', flush=True)
len(rows)

In [ ]:
positive_rows = [row for row in rows if row['기대 대상']]
llm_positive_rows = [row for row in rows if row['LLM 호출'] and row['LLM 기대 행동'] == 'candidate']
abstain_rows = [row for row in rows if row['LLM 기대 행동'] == 'abstain']
llm_rows = [row for row in rows if row['LLM 호출']]
validity_values = [value for row in llm_rows for value in row['원본 검색어 exact 유효 여부']]
latencies = [row['LLM 호출 시간(ms)'] for row in llm_rows if row['LLM 호출 시간(ms)'] is not None]

def p95(values):
    return sorted(values)[ceil(len(values) * 0.95) - 1] if values else None

summary = {
    '평가 사례 수': len(rows),
    'A. 기준선 Candidate Recall': sum(row['기준선 정답 포함'] for row in positive_rows) / len(positive_rows),
    'C. 개선 프롬프트 Candidate Recall': sum(row['확장 후 정답 포함'] for row in positive_rows) / len(positive_rows),
    'C. LLM Candidate Recall': sum(row['LLM 정답 포함'] for row in llm_positive_rows) / len(llm_positive_rows),
    'C. LLM Expansion Contribution': sum(row['LLM 신규 기여'] for row in positive_rows) / len(positive_rows),
    'C. Abstention Accuracy': sum(row['안전 보류'] for row in abstain_rows) / len(abstain_rows),
    'C. Structured-output Success Rate': sum(row['구조화 출력 성공'] for row in llm_rows) / len(llm_rows),
    'C. Search-term Validity': sum(validity_values) / len(validity_values) if validity_values else None,
    'C. LLM Call Rate': len(llm_rows) / len(rows),
    'C. LLM p50(ms)': median(latencies) if latencies else None,
    'C. LLM p95(ms)': p95(latencies),
}
summary

In [ ]:
import pandas as pd

columns = [
    '원문 기업 표현', 'LLM 기대 행동', 'LLM 호출', 'LLM 결정', 'LLM 원본 검색어', '실제 탐색 검색어',
    '기준선 정답 포함', 'LLM 정답 포함', '확장 후 정답 포함', 'LLM 신규 기여',
    '안전 보류', '원본 검색어 exact 유효 여부', 'LLM 호출 시간(ms)', '오류',
]
display(pd.DataFrame(rows)[columns])

## 6. 해석 기준

- C의 Candidate Recall 또는 LLM Expansion Contribution이 A보다 높아야 LLM 검색어 확장을 유지할 근거가 생긴다.
- C의 Abstention Accuracy가 낮으면 포괄 표현·존재하지 않는 이름을 실제 기업으로 무리하게 연결할 위험이 있으므로 미채택한다.
- C가 13번 B보다 검색어 회수와 안전성을 개선해도, 이번 실행만으로 프롬프트 정책이 원인이라고 단정하지 않는다. 같은 데이터 스냅샷·실행 환경에서 B/C를 다시 실행해 우위를 확인한 뒤에만 프롬프트 효과로 해석한다.
- C가 여전히 이득이 없으면, Qwen 모델 또는 LLM 검색어 확장 자체를 API 모델 비교로 검증해야 한다.
- latency는 결과 보고서에 기록하지만, 이 로컬 평가에서 품질·안전성 판단보다 우선하지 않는다.

실행 결과를 저장한 뒤 알려주면, 13번 B와 14번 C를 비교해 HTML 보고서에 채택·미채택 근거를 기록한다.